# 04 — Drift Analysis

Computes the **temporal diagnostic profile** for each window pair (A, B) across all
supported model types, then renders a combined dashboard. Reflects the methodology
described in the thesis (no drift ratio; per-window baselines reported separately
for A and B; global attribution drift restricted to SHAP).

**Model types:** `MODEL_TYPES = ['xgboost', 'logreg', 'mlp_plr']`
Artifacts are read from `data/models/{model_type}/`, `data/shap/{model_type}/`, and
`data/lime/{model_type}/`.

**Input:** `data/processed/`, `data/windows/`, `data/models/{model_type}/`, `data/shap/{model_type}/`, `data/lime/{model_type}/`
**Output per model type:** `data/results/{model_type}/drift_metrics.csv`
**Combined outputs:** `data/results/combined/drift_metrics_combined.csv`, `data/results/combined/drift_dashboard_*.png`

---

| Metric | Symbol | XGBoost (SHAP) | XGBoost / MLP-PLR (LIME) | LR (coef ref) |
|---|---|---|---|---|
| Covariate drift | Δ_X | ✓ | ✓ | ✓ |
| Target drift | Δ_Y | ✓ | ✓ | ✓ |
| Performance change | Δ_perf | ✓ | ✓ | ✓ |
| Local dynamic drift (cosine) | loc_cos | ✓ (SHAP) | ✓ (LIME) | — |
| Local dynamic drift (RBO) | loc_rbo | ✓ (SHAP) | ✓ (LIME) | — |
| Within-window baseline (cosine) — A | base_cos_A | ✓ | ✓ | ✓ |
| Within-window baseline (cosine) — B | base_cos_B | ✓ | ✓ | ✓ |
| Within-window baseline (RBO) — A | base_rbo_A | ✓ | ✓ | ✓ |
| Within-window baseline (RBO) — B | base_rbo_B | ✓ | ✓ | ✓ |
| Global attribution drift | global_rbo | ✓ (SHAP only) | — | — |
| Explainer stochasticity | `explainer_stoch_max_abs` / `explainer_stoch_median_abs`  | ✓ | ✓ | NaN (deterministic) |

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


> **Setup note:** this notebook requires the `rbo` package for rank-biased overlap metrics (§3.5, §3.8).  
> If not already installed, run: `pip install rbo`

In [4]:
%pip install rbo

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import StandardScaler
from itertools import combinations
import rbo

WORKSPACE  = Path('/content/drive/MyDrive/Thesis/Shoppers_workspace')
PROC_DIR   = WORKSPACE / 'data' / 'processed'
WIN_DIR    = WORKSPACE / 'data' / 'windows'

# ── Per-model-type base directories ───────────────────────────────────
MODEL_DIR_BASE   = WORKSPACE / 'data' / 'models'
SHAP_DIR_BASE    = WORKSPACE / 'data' / 'shap'
LIME_DIR_BASE    = WORKSPACE / 'data' / 'lime'
RESULTS_DIR_BASE = WORKSPACE / 'data' / 'results'

# Aliases used by downstream cells
SHAP_DIR    = SHAP_DIR_BASE / 'xgboost'
RESULTS_DIR = RESULTS_DIR_BASE

COMBINED_DIR = RESULTS_DIR_BASE / 'combined'
COMBINED_DIR.mkdir(parents=True, exist_ok=True)

# ── What to compute ────────────────────────────────────────────────────
MODEL_TYPES = ['xgboost', 'logreg', 'mlp_plr']
EXPLAINERS  = ['shap', 'lime']   # per-row attribution sources to iterate over
                                 # LR emits a 'coef' row handled outside this list

RBO_P = 0.9

print('Imports OK')
print(f'MODEL_TYPES: {MODEL_TYPES}')
print(f'EXPLAINERS : {EXPLAINERS}')

In [ ]:
X    = pd.read_parquet(PROC_DIR / 'X.parquet').values.astype(np.float32)
Y    = np.load(PROC_DIR / 'Y.npy').astype(np.int8)

with open(PROC_DIR / 'feature_names.json') as f:
    feature_names_json = json.load(f)
feature_names     = feature_names_json['all']   # flat list, 119 names in column order
num_feature_names = feature_names_json['num']   # 113 numeric feature names

# Column indices of numeric / binary features in X (column order matches feature_names)
num_col_idx = [feature_names.index(fn) for fn in num_feature_names]
bin_col_idx = [i for i in range(len(feature_names)) if i not in set(num_col_idx)]

with open(WIN_DIR / 'window_config.json') as f:
    config = json.load(f)

R     = config['parameters']['R']
pairs = config['pairs']
n_features = len(feature_names)

assert X.shape[0] == Y.shape[0], "X and Y have inconsistent numbers of rows."
assert X.shape[1] == n_features, "X column count does not match feature_names['all']."
assert len(set(feature_names)) == len(feature_names), "feature_names contains duplicates."
assert len(set(num_col_idx) & set(bin_col_idx)) == 0, "Numeric and binary indices overlap."
assert len(num_col_idx) + len(bin_col_idx) == n_features, (
    "Numeric/binary feature partition does not cover all features."
)
assert len(pairs) > 0, "No rolling-window pairs found in window_config.json."

print(f'X: {X.shape}, features: {n_features}, R={R}, pairs: {len(pairs)}')
print(f'  Numeric: {len(num_col_idx)}, Binary: {len(bin_col_idx)}')

X: (160057, 119), features: 119, R=8, pairs: 2
  Numeric: 113, Binary: 6


## Distance and RBO helper functions

In [ ]:
def cosine_distance(u: np.ndarray, v: np.ndarray) -> float:
    """d_cos(u,v) = 1 - cosine_similarity(u,v).
    Both-zero → 0.0; one-zero one-nonzero → np.nan (excluded by nanmedian aggregation).
    """
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    if norm_u < 1e-12 and norm_v < 1e-12:
        return 0.0
    if norm_u < 1e-12 or norm_v < 1e-12:
        # One attribution vector is exactly zero; the other is not.
        # Not a meaningful distance — marked nan and dropped during aggregation.
        return np.nan
    return float(1.0 - np.dot(u, v) / (norm_u * norm_v))


def rbo_score_local(l1, l2, p=0.9):
    """Simple RBO implementation as fallback."""
    if not l1 or not l2: return 0.0
    n = min(len(l1), len(l2))
    s1, s2 = set(), set()
    agreement = 0.0
    res = 0.0
    for i in range(n):
        s1.add(l1[i])
        s2.add(l2[i])
        agreement = len(s1.intersection(s2)) / (i + 1)
        res += agreement * (p ** i)
    return res * (1 - p)


def rbo_distance(u: np.ndarray, v: np.ndarray, p: float = RBO_P) -> float:
    """
    d_rbo(u, v) = 1 - RBO(rank(u), rank(v)).
    Features are ranked by decreasing absolute attribution.
    Both-zero → 0.0; one-zero one-nonzero → np.nan.
    """
    u_zero = np.all(np.abs(u) < 1e-12)
    v_zero = np.all(np.abs(v) < 1e-12)
    if u_zero and v_zero:
        return 0.0
    if u_zero or v_zero:
        return np.nan
    rank_u = list(np.argsort(-np.abs(u)))
    rank_v = list(np.argsort(-np.abs(v)))
    score = rbo.RankingSimilarity(rank_u, rank_v).rbo(p=p)
    return float(1.0 - score)


def instance_dynamic_drift(
    phi_A: np.ndarray,  # (R, p) — R attribution vectors from window A
    phi_B: np.ndarray,  # (R, p) — R attribution vectors from window B
    dist_fn,
) -> float:
    """
    δ_dyn(x; d) = mean over all R×R cross-window pairs of d(φ_A^{(r)}, φ_B^{(s)}).
    """
    dists = []
    for r in range(R):
        for s in range(R):
            dists.append(dist_fn(phi_A[r], phi_B[s]))
    return float(np.nanmean(dists))


def instance_baseline_instability(
    phi: np.ndarray,  # (R, p) — R attribution vectors from same window
    dist_fn,
) -> float:
    """
    δ^{base}(x; d) = median over all r < r' pairs of d(φ^{(r)}, φ^{(r')}).
    """
    dists = []
    for r, r2 in combinations(range(R), 2):
        dists.append(dist_fn(phi[r], phi[r2]))
    return float(np.nanmedian(dists)) if dists else 0.0


def rbo_global_drift(
    phi_bar_A: np.ndarray,  # (|F|, p)
    phi_bar_B: np.ndarray,  # (|F|, p)
    p: float = RBO_P,
) -> float:
    """
    Δ_E^{glob}: 1 − RBO(rank(g_A), rank(g_B)) where g(j) = mean_x |φ̄_j(x)|.
    """
    g_A = np.abs(phi_bar_A).mean(axis=0)
    g_B = np.abs(phi_bar_B).mean(axis=0)
    rank_A = list(np.argsort(-g_A))
    rank_B = list(np.argsort(-g_B))
    score = rbo.RankingSimilarity(rank_A, rank_B).rbo(p=p)
    return float(1.0 - score)


print('Distance functions defined.')

Distance functions defined.


## Main drift computation loop

For each window pair we compute all metrics and append to a results list.

## Logistic Regression — reference treatment

LR is treated as a transparent reference, *not* as a post-hoc explanation drift
stream. Two concrete artefacts are used:

- **Replica coefficient tensors** (`coef_A.npy`, `coef_B.npy`, shape `(R, p)`)
  are used only for within-window coefficient instability under cosine and RBO
  distance. This estimates the variability induced by the retraining procedure
  for the linear model and plays the same role as `base_*_A` / `base_*_B`
  for SHAP and LIME.
- **Full-window coefficient vectors** (`coef_A_full.npy`, `coef_B_full.npy`,
  shape `(p,)`) are obtained from a single deterministic LR fit on the full
  training window, with no bootstrap and no replica averaging. These are
  visualised side by side as a transparent A-vs-B reference (see the
  full-window coefficient comparison cell below).

| Attribution metric              | LR analogue                                                   | Notes                                       |
|---------------------------------|---------------------------------------------------------------|---------------------------------------------|
| Covariate drift Δ_X             | ✓ identical                                                   | data-level, model-agnostic                  |
| Target drift Δ_Y                | ✓ identical                                                   | data-level, model-agnostic                  |
| Performance change Δ_perf       | ✓ identical                                                   | same PR-AUC formula                         |
| `loc_cos` / `loc_rbo`           | — *(not computed)*                                            | replica-coefficient cross-window drift conflates retraining noise with the per-replica scaler basis; not reported |
| `base_cos_A/B`, `base_rbo_A/B`  | median pairwise distance over C(R,2) replica coefficient pairs per window | within-window coefficient instability       |
| Global attribution drift        | — *(not computed)*                                            | LR coefficients are global by construction; covered by the full-window comparison plot |
| `explainer_stoch_max_abs` / `explainer_stoch_median_abs`              | NaN                                                           | LR coefficient solvers are deterministic    |

In [ ]:
all_results = {}


def _load_attributions(explainer_name, pair_dir, attr_dir, pred_data):
    """Return (attr_A, attr_B, flagged_subset_idx_within_flagged) or None.

    For SHAP:
    - XGBoost usually runs on the full flagged set, so no subset file exists.
    - MLP-PLR may run on a saved SHAP subsample. If
      mlp_shap_subsample_idx.npy exists, it is used as the subset index.
      Otherwise, SHAP is treated as full-flagged for backwards compatibility.

    For LIME:
    - attr files are (R, |F_lime|, p);
    - lime_flagged_idx.npy stores positions within the full flagged set.
    """
    if explainer_name == 'shap':
        a_path = attr_dir / 'shap_A.npy'
        b_path = attr_dir / 'shap_B.npy'
        subset_path = attr_dir / 'mlp_shap_subsample_idx.npy'

        if not (a_path.exists() and b_path.exists()):
            return None

        a = np.load(a_path)
        b = np.load(b_path)

        if subset_path.exists():
            subset = np.load(subset_path).astype(np.int64)
        else:
            subset = np.arange(a.shape[1], dtype=np.int64)

        return a, b, subset

    elif explainer_name == 'lime':
        a_path      = attr_dir / 'lime_A.npy'
        b_path      = attr_dir / 'lime_B.npy'
        subset_path = attr_dir / 'lime_flagged_idx.npy'

        if not (a_path.exists() and b_path.exists() and subset_path.exists()):
            return None

        a = np.load(a_path)
        b = np.load(b_path)
        subset = np.load(subset_path).astype(np.int64)

        return a, b, subset

    raise ValueError(f'Unknown explainer: {explainer_name}')

def _validate_attribution_shapes(attr_A, attr_B, subset, flagged_local_idx,
                                 mt, explainer_name, pid):
    """Fail early if attribution arrays are stale or incompatible."""
    assert attr_A.ndim == 3 and attr_B.ndim == 3, (
        f'{mt}/{explainer_name}/pair {pid:02d}: attribution arrays must be 3-D.'
    )

    assert attr_A.shape == attr_B.shape, (
        f'{mt}/{explainer_name}/pair {pid:02d}: attr_A and attr_B shape mismatch: '
        f'{attr_A.shape} vs {attr_B.shape}'
    )

    assert attr_A.shape[0] == R, (
        f'{mt}/{explainer_name}/pair {pid:02d}: expected R={R}, '
        f'got attribution R={attr_A.shape[0]}'
    )

    assert attr_A.shape[2] == n_features, (
        f'{mt}/{explainer_name}/pair {pid:02d}: expected {n_features} features, '
        f'got {attr_A.shape[2]}'
    )

    assert len(subset) == attr_A.shape[1], (
        f'{mt}/{explainer_name}/pair {pid:02d}: subset length {len(subset)} '
        f'does not match attribution instances {attr_A.shape[1]}'
    )

    assert np.all((subset >= 0) & (subset < len(flagged_local_idx))), (
        f'{mt}/{explainer_name}/pair {pid:02d}: subset indices are outside flagged_idx.'
    )

def _validate_predictions(pred_data, mt, pid, idx_eval):
    """Check that predictions.npz contains the keys and shapes expected by notebook 04."""
    required_keys = [
        'preds_A', 'preds_B',
        'p_hat_A', 'p_hat_B',
        'flagged_idx',
        'Y_eval',
        'pr_auc_A', 'pr_auc_B',
        'per_replica_pr_auc_A', 'per_replica_pr_auc_B',
    ]

    missing = [k for k in required_keys if k not in pred_data.files]
    assert not missing, (
        f'{mt}/pair {pid:02d}: predictions.npz missing keys: {missing}. '
        'Delete stale predictions and rerun the training notebook.'
    )

    flagged_idx = pred_data['flagged_idx']
    assert pred_data['preds_A'].shape == pred_data['preds_B'].shape, (
        f'{mt}/pair {pid:02d}: preds_A and preds_B shape mismatch.'
    )

    assert pred_data['preds_A'].shape[0] == R, (
        f'{mt}/pair {pid:02d}: expected R={R}, got {pred_data["preds_A"].shape[0]}.'
    )

    assert pred_data['preds_A'].shape[1] == len(idx_eval), (
        f'{mt}/pair {pid:02d}: prediction length does not match idx_eval.'
    )

    assert np.all((flagged_idx >= 0) & (flagged_idx < len(idx_eval))), (
        f'{mt}/pair {pid:02d}: flagged_idx contains positions outside idx_eval.'
    )


def _attribution_drift_metrics(attr_A: np.ndarray, attr_B: np.ndarray) -> dict:
    """Compute local drift, separate A/B baselines, and global drift.

    Returns keys: loc_cos, loc_rbo, base_cos_A, base_cos_B, base_rbo_A,
    base_rbo_B, global_rbo. The caller sets global_rbo to NaN for LIME.
    """
    n_sub = attr_A.shape[1]
    out   = {}

    # Local dynamic cross-window drift
    dyn_cos_per_instance = []
    dyn_rbo_per_instance = []
    for i in range(n_sub):
        phi_A_i = attr_A[:, i, :]
        phi_B_i = attr_B[:, i, :]
        dyn_cos_per_instance.append(instance_dynamic_drift(phi_A_i, phi_B_i, cosine_distance))
        dyn_rbo_per_instance.append(instance_dynamic_drift(phi_A_i, phi_B_i, rbo_distance))
    out['loc_cos'] = float(np.nanmedian(dyn_cos_per_instance))
    out['loc_rbo'] = float(np.nanmedian(dyn_rbo_per_instance))

    # Within-window baseline instability — reported separately for A and B
    base_cos_A_per, base_cos_B_per = [], []
    base_rbo_A_per, base_rbo_B_per = [], []
    for i in range(n_sub):
        phi_A_i = attr_A[:, i, :]
        phi_B_i = attr_B[:, i, :]
        base_cos_A_per.append(instance_baseline_instability(phi_A_i, cosine_distance))
        base_cos_B_per.append(instance_baseline_instability(phi_B_i, cosine_distance))
        base_rbo_A_per.append(instance_baseline_instability(phi_A_i, rbo_distance))
        base_rbo_B_per.append(instance_baseline_instability(phi_B_i, rbo_distance))
    out['base_cos_A'] = float(np.nanmedian(base_cos_A_per))
    out['base_cos_B'] = float(np.nanmedian(base_cos_B_per))
    out['base_rbo_A'] = float(np.nanmedian(base_rbo_A_per))
    out['base_rbo_B'] = float(np.nanmedian(base_rbo_B_per))

    # Global attribution drift (SHAP only — caller NaNs this for LIME)
    phi_bar_A = attr_A.mean(axis=0)
    phi_bar_B = attr_B.mean(axis=0)
    out['global_rbo'] = rbo_global_drift(phi_bar_A, phi_bar_B)

    return out


def _make_base_row(mt, pid, p, pred_data, idx_A, idx_B, flagged_local_idx):
    """Per-pair metrics that are explainer-independent (data-drift / target-drift / perf)."""
    row = {
        'model_type':    mt,
        'explainer':     None,   # filled in later
        'pair_id':       pid,
        'step_label_A':  p['step_label_A'],
        'step_label_B':  p['step_label_B'],
        'n_train_A':     p['n_train_A'],
        'n_train_B':     p['n_train_B'],
        'n_eval':        p['n_eval'],
        'n_flagged':     len(flagged_local_idx),
        'pr_auc_A':      float(pred_data['pr_auc_A']),
        'pr_auc_B':      float(pred_data['pr_auc_B']),
    }

    # Performance change: PR-AUC_B − PR-AUC_A (positive = newer model better)
    row['delta_perf'] = float(pred_data['pr_auc_B']) - float(pred_data['pr_auc_A'])

    # Target drift (binary): absolute difference in positive-class rate
    row['delta_Y'] = abs(float(Y[idx_A].mean()) - float(Y[idx_B].mean()))

    # Covariate drift: feature-wise W1 in the reference frame, averaged
    pair_dir = MODEL_DIR_BASE / mt / f'pair_{pid:02d}'
    reference_scaler_path = pair_dir / 'reference_scaler.joblib'
    if reference_scaler_path.exists():
        reference_scaler = joblib.load(reference_scaler_path)
        X_A_num_sc = reference_scaler.transform(X[idx_A][:, num_col_idx])
        X_B_num_sc = reference_scaler.transform(X[idx_B][:, num_col_idx])
        w1_per_feat = np.zeros(n_features, dtype=np.float64)
        for k, j in enumerate(num_col_idx):
            w1_per_feat[j] = wasserstein_distance(X_A_num_sc[:, k], X_B_num_sc[:, k])
        for j in bin_col_idx:
            w1_per_feat[j] = wasserstein_distance(X[idx_A][:, j], X[idx_B][:, j])
        row['delta_X'] = float(w1_per_feat.mean())
    else:
        row['delta_X'] = np.nan

    return row


# ═════════════════════════════════════════════════════════════════════════════
# Main loop
# ═════════════════════════════════════════════════════════════════════════════

for mt in MODEL_TYPES:
    print(f'\n{"="*60}')
    print(f'  Model type: {mt}')
    print(f'{"="*60}')

    mt_model_dir = MODEL_DIR_BASE / mt
    results = []

    for p in pairs:
        pid      = p['pair_id']
        pair_dir = mt_model_dir / f'pair_{pid:02d}'

        print(f'\n── [{mt}] Pair {pid:02d}: A_end={p["step_label_A"]}  B_end={p["step_label_B"]} ──')

        pred_data_path = pair_dir / 'predictions.npz'
        if not pred_data_path.exists():
            print(f'  WARNING: predictions.npz not found for pair {pid:02d}. Skipping.')
            continue

        pred_data = np.load(pred_data_path)
        idx_A    = np.array(p['idx_A'],    dtype=np.int64)
        idx_B    = np.array(p['idx_B'],    dtype=np.int64)
        idx_eval = np.array(p['idx_eval'], dtype=np.int64)
        _validate_predictions(pred_data, mt, pid, idx_eval)

        flagged_local_idx = pred_data['flagged_idx']
        n_flagged = len(flagged_local_idx)
        print(f'  Flagged: {n_flagged:,}')

        base_row = _make_base_row(mt, pid, p, pred_data, idx_A, idx_B, flagged_local_idx)
        print(f'  Perf: Δ={base_row["delta_perf"]:+.4f}   Target drift: {base_row["delta_Y"]:.4f}   Cov drift: {base_row["delta_X"]:.4f}')

        # ─────────────────────────────────────────────────────────────
        # LR reference path — within-window coefficient instability only.
        # These distances are computed on the saved replica coefficient vectors.
        # Because each replica includes its own fitted scaler, this should be read as
        # LR pipeline-level coefficient instability rather than pure coefficient
        # variation in one fixed standardized basis.
        # ─────────────────────────────────────────────────────────────
        if mt == 'logreg':
            coef_A_path = pair_dir / 'coef_A.npy'
            coef_B_path = pair_dir / 'coef_B.npy'
            if coef_A_path.exists() and coef_B_path.exists():
                coef_A = np.load(coef_A_path)
                coef_B = np.load(coef_B_path)

                row = dict(base_row)
                row['explainer']                   = 'coef'
                row['n_subset']                    = R   # one coef vector per replica
                row['shap_stochasticity']          = np.nan
                row['explainer_stoch_max_abs']     = np.nan
                row['explainer_stoch_median_abs']  = np.nan
                row['explainer_is_deterministic']  = True

                # Cross-window drift columns are intentionally NaN for LR.
                row['loc_cos']    = np.nan
                row['loc_rbo']    = np.nan
                row['global_rbo'] = np.nan

                # Within-window coefficient instability — separate for A and B.
                row['base_cos_A'] = float(np.nanmedian([cosine_distance(coef_A[r], coef_A[r2])
                                                        for r, r2 in combinations(range(R), 2)]))
                row['base_cos_B'] = float(np.nanmedian([cosine_distance(coef_B[r], coef_B[r2])
                                                        for r, r2 in combinations(range(R), 2)]))
                row['base_rbo_A'] = float(np.nanmedian([rbo_distance(coef_A[r], coef_A[r2])
                                                        for r, r2 in combinations(range(R), 2)]))
                row['base_rbo_B'] = float(np.nanmedian([rbo_distance(coef_B[r], coef_B[r2])
                                                        for r, r2 in combinations(range(R), 2)]))

                print(f'  [coef]   baseA_cos={row["base_cos_A"]:.4f}  baseB_cos={row["base_cos_B"]:.4f}  '
                      f'baseA_rbo={row["base_rbo_A"]:.4f}  baseB_rbo={row["base_rbo_B"]:.4f}')
                results.append(row)
            else:
                print(f'  WARNING: coef_A.npy/coef_B.npy not found for LR pair {pid:02d}')

            
        # ─────────────────────────────────────────────────────────────
        # Explainer-based rows — one per explainer in EXPLAINERS
        # ─────────────────────────────────────────────────────────────
        for explainer_name in (EXPLAINERS if mt != 'logreg' else []):

            if explainer_name == 'shap':
                attr_dir = SHAP_DIR_BASE / mt / f'pair_{pid:02d}'
            elif explainer_name == 'lime':
                attr_dir = LIME_DIR_BASE / mt / f'pair_{pid:02d}'
            else:
                print(f'  WARNING: unknown explainer {explainer_name!r}')
                continue

            loaded = _load_attributions(explainer_name, pair_dir, attr_dir, pred_data)
            if loaded is None:
                print(f'  WARNING: {explainer_name} files not found for pair {pid:02d}. Skipping.')
                continue
            attr_A, attr_B, subset = loaded

            _validate_attribution_shapes(
                attr_A, attr_B, subset, flagged_local_idx,
                mt, explainer_name, pid
            )

            row = dict(base_row)
            row['explainer'] = explainer_name
            row['n_subset']  = int(len(subset))

            stoch_path = attr_dir / 'stochasticity.json'
            if stoch_path.exists():
                with open(stoch_path) as f:
                    stoch_data = json.load(f)

                row['explainer_stoch_max_abs'] = float(stoch_data.get('max_abs_diff', np.nan))
                row['explainer_stoch_median_abs'] = float(stoch_data.get('median_abs_diff', np.nan))
                row['explainer_is_deterministic'] = bool(stoch_data.get('is_deterministic', False))

                # Backwards-compatible alias for older plotting code / old result files.
                row['shap_stochasticity'] = row['explainer_stoch_max_abs']
            else:
                row['explainer_stoch_max_abs'] = np.nan
                row['explainer_stoch_median_abs'] = np.nan
                row['explainer_is_deterministic'] = np.nan
                row['shap_stochasticity'] = np.nan
                print(f'  WARNING: stochasticity.json not found under {attr_dir.name}')

            if attr_A.shape[1] == 0:
                for k in ['loc_cos', 'loc_rbo',
                          'base_cos_A', 'base_cos_B', 'base_rbo_A', 'base_rbo_B',
                          'global_rbo']:
                    row[k] = np.nan
                print(f'  [{explainer_name}] No subset instances — drift metrics NaN.')
                results.append(row)
                continue

            metrics = _attribution_drift_metrics(attr_A, attr_B)
            # Global drift is reported for SHAP only. LIME's local-surrogate
            # construction makes a mean-|attribution| ranking less directly
            # meaningful, so it is intentionally not reported.
            if explainer_name == 'lime':
                metrics['global_rbo'] = np.nan
            row.update(metrics)

            print(f'  [{explainer_name:<4s}]  loc_cos={row["loc_cos"]:.4f}  baseA={row["base_cos_A"]:.4f}  baseB={row["base_cos_B"]:.4f}  '
                  f'loc_rbo={row["loc_rbo"]:.4f}  global_rbo={row["global_rbo"]:.4f}  '
                  f'stoch={row["explainer_stoch_max_abs"]:.2e}')

            results.append(row)

    all_results[mt] = results

print('\n✓ All drift metrics computed.')

## Save results

In [ ]:
# ── Save per-model-type CSVs (all explainers combined per model type) ─────────
for mt in MODEL_TYPES:
    if mt not in all_results or not all_results[mt]:
        continue
    mt_results_dir = RESULTS_DIR_BASE / mt
    mt_results_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(all_results[mt]).to_csv(mt_results_dir / 'drift_metrics.csv', index=False)
    print(f'Saved {mt}/drift_metrics.csv  ({len(all_results[mt])} rows)')

# ── Combined CSV ──────────────────────────────────────────────────────────────
result_frames = [
    pd.DataFrame(all_results[mt])
    for mt in MODEL_TYPES
    if mt in all_results and all_results[mt]
]

if result_frames:
    combined_df = pd.concat(result_frames, ignore_index=True)
    combined_df.to_csv(COMBINED_DIR / 'drift_metrics_combined.csv', index=False)
    print(f'Saved combined/drift_metrics_combined.csv  ({len(combined_df)} rows)')
else:
    combined_df = pd.DataFrame()
    print('No result rows found; combined CSV not written.')

    
# ── Display summary per (model_type, explainer) ───────────────────────────────
if not combined_df.empty:
    print('\n── Rows per (model_type, explainer) ──')
    print(combined_df.groupby(['model_type', 'explainer']).size().to_string())

    print('\n── Per-pair summary (xgboost, shap) ──')
    sub = combined_df.query("model_type == 'xgboost' and explainer == 'shap'")
    if not sub.empty:
        cols = [c for c in [
            'pair_id', 'step_label_A', 'step_label_B',
            'pr_auc_A', 'pr_auc_B', 'delta_perf', 'delta_X', 'delta_Y',
            'loc_cos', 'base_cos_A', 'base_cos_B',
            'loc_rbo', 'base_rbo_A', 'base_rbo_B',
            'global_rbo', 'explainer_stoch_max_abs', 'explainer_stoch_median_abs',
        ] if c in sub.columns]
        print(sub[cols].to_string(index=False))

# ── Backwards-compat alias ─────────────────────────────────────────────────────
drift_df = combined_df.query("model_type == 'xgboost' and explainer == 'shap'").copy() \
           if not combined_df.empty else pd.DataFrame()


## Temporal diagnostic dashboard

Renders the diagnostic profile described in the methodology
(`subsec:reporting_across_time`) as one figure per explainer. Each dashboard
shows contextual signals (Δ_X, Δ_Y, Δ_perf), local drift vs separate within-window
baselines for both A and B (cosine and RBO, one panel per model), global SHAP
drift (where defined), and the explainer stochasticity diagnostic.

Set `REPORT_EXPLAINER` to `'shap'` or `'lime'` and re-run this cell to switch dashboards. Logistic regression is handled separately as a transparent coefficient reference.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Temporal diagnostic dashboard
#
# Reports the diagnostic profile: contextual signals, local drift vs separate
# within-window baselines (A and B), global SHAP drift, and explainer
# stochasticity, all side by side per post-hoc explainer / model-family.
#
# Set REPORT_EXPLAINER to 'shap' or 'lime'.
#
# Logistic regression is a transparent reference and does not appear in the
# explanation-drift panels. It contributes to the contextual panels (Δ_X, Δ_Y,
# Δ_perf) only. The full-window LR coefficient comparison is rendered by a
# separate cell.
# ─────────────────────────────────────────────────────────────────────────

REPORT_EXPLAINER = 'shap'

df_all = combined_df.copy()
if REPORT_EXPLAINER == 'shap':
    df_all = df_all[df_all['explainer'] == 'shap']
elif REPORT_EXPLAINER == 'lime':
    df_all = df_all[df_all['explainer'] == 'lime']
else:
    raise ValueError(f'Unknown REPORT_EXPLAINER: {REPORT_EXPLAINER!r}')

df_all = df_all[df_all['model_type'] != 'logreg']
df_all = df_all.dropna(subset=['loc_cos'])

if df_all.empty:
    print(f'No rows for explainer={REPORT_EXPLAINER!r} — skipping dashboard.')
else:
    pair_ids = sorted(df_all['pair_id'].unique())
    x_idx    = list(range(len(pair_ids)))

    # x-axis labels — pair_id → step_label_A from the union (first match)
    label_lookup = (combined_df.drop_duplicates('pair_id')
                              .set_index('pair_id')['step_label_A'].to_dict())
    x_labels = [label_lookup.get(pid, str(pid)) for pid in pair_ids]

    COLORS  = {'xgboost': 'crimson',  'logreg': 'steelblue', 'mlp_plr': 'darkviolet'}
    MARKERS = {'xgboost': 'o',        'logreg': 's',         'mlp_plr': '^'}
    LINES   = {'xgboost': '-',        'logreg': '--',        'mlp_plr': '-.'}
    PRESENT = [mt for mt in MODEL_TYPES if mt in df_all['model_type'].unique()]

    def _x_local(dft):
        """Map a per-model dataframe's pair_id column to positions on the union x-axis."""
        return [pair_ids.index(pid) for pid in dft['pair_id'].tolist()]

    fig = plt.figure(figsize=(15, 17))
    gs  = gridspec.GridSpec(4, 3, height_ratios=[1, 1.4, 1.4, 1],
                            hspace=0.6, wspace=0.32)

    # ── Row 0: contextual signals ─────────────────────────────────────
    ax_dx   = fig.add_subplot(gs[0, 0])
    ax_dy   = fig.add_subplot(gs[0, 1])
    ax_perf = fig.add_subplot(gs[0, 2])

    # Δ_X and Δ_Y do not depend on model_type — pull from any present model
    df_ref = df_all[df_all['model_type'] == PRESENT[0]].sort_values('pair_id')
    xr = _x_local(df_ref)
    ax_dx.plot(xr, df_ref['delta_X'].values, 'o-', color='steelblue')
    ax_dx.set_title('Covariate drift (Δ_X)')
    ax_dx.set_ylabel('Mean Wasserstein')

    ax_dy.plot(xr, df_ref['delta_Y'].values, 's-', color='darkorange')
    ax_dy.set_title('Target drift (Δ_Y)')
    ax_dy.set_ylabel('|p+_A − p+_B|')

    # Δ_perf is per-model — pull from the union frame (combined_df) so LR's
    # reference performance line still appears in the context panel even
    # though it does not contribute to the explanation-drift panels below.
    perf_df = (combined_df
               .drop_duplicates(['model_type', 'pair_id'])
               .sort_values('pair_id'))
    perf_present = [mt for mt in MODEL_TYPES if mt in perf_df['model_type'].unique()]
    for mt in perf_present:
        dft = perf_df[perf_df['model_type'] == mt]
        ax_perf.plot(_x_local(dft), dft['delta_perf'].values,
                     MARKERS[mt] + LINES[mt], color=COLORS[mt],
                     label=mt, linewidth=1.6)
    ax_perf.axhline(0.0, color='grey', linewidth=0.7, linestyle=':')
    ax_perf.set_title('Performance change (Δ_perf = PR-AUC_B − PR-AUC_A)')
    ax_perf.set_ylabel('Δ PR-AUC')
    ax_perf.legend(fontsize=8)

    for ax in (ax_dx, ax_dy, ax_perf):
        ax.set_xticks(x_idx)
        ax.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=7)

    # ── Row 1: cosine — local drift vs A/B baselines per model ────────
    for k, mt in enumerate(MODEL_TYPES):
        ax = fig.add_subplot(gs[1, k])
        dft = df_all[df_all['model_type'] == mt].sort_values('pair_id')
        if dft.empty:
            ax.set_title(f'{mt} — cosine (no data)')
            ax.set_xticks([])
            continue
        xl = _x_local(dft)
        ax.plot(xl, dft['loc_cos'].values, 'o-', color=COLORS[mt],
                linewidth=2.0, label='Local drift')
        ax.plot(xl, dft['base_cos_A'].values, 's--', color='dimgray',
                alpha=0.85, linewidth=1.2, label='Baseline A')
        ax.plot(xl, dft['base_cos_B'].values, '^:',  color='black',
                alpha=0.85, linewidth=1.2, label='Baseline B')
        ax.set_title(f'{mt} — local drift vs baselines (cosine)')
        ax.set_ylabel('Cosine distance')
        ax.legend(fontsize=7, loc='best')
        ax.set_xticks(x_idx)
        ax.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=7)

    # ── Row 2: RBO — local drift vs A/B baselines per model ───────────
    for k, mt in enumerate(MODEL_TYPES):
        ax = fig.add_subplot(gs[2, k])
        dft = df_all[df_all['model_type'] == mt].sort_values('pair_id')
        if dft.empty:
            ax.set_title(f'{mt} — RBO (no data)')
            ax.set_xticks([])
            continue
        xl = _x_local(dft)
        ax.plot(xl, dft['loc_rbo'].values, 'o-', color=COLORS[mt],
                linewidth=2.0, label='Local drift')
        ax.plot(xl, dft['base_rbo_A'].values, 's--', color='dimgray',
                alpha=0.85, linewidth=1.2, label='Baseline A')
        ax.plot(xl, dft['base_rbo_B'].values, '^:',  color='black',
                alpha=0.85, linewidth=1.2, label='Baseline B')
        ax.set_title(f'{mt} — local drift vs baselines (RBO)')
        ax.set_ylabel('RBO distance (1 − RBO)')
        ax.legend(fontsize=7, loc='best')
        ax.set_xticks(x_idx)
        ax.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=7)

    # ── Row 3: global drift (left, wider) + explainer stochasticity ───
    ax_glob  = fig.add_subplot(gs[3, 0:2])
    ax_stoch = fig.add_subplot(gs[3, 2])

    glob_plotted = False
    for mt in PRESENT:
        dft = df_all[df_all['model_type'] == mt].sort_values('pair_id')
        if 'global_rbo' not in dft.columns or dft['global_rbo'].isna().all():
            continue
        ax_glob.plot(_x_local(dft), dft['global_rbo'].values,
                     MARKERS[mt] + LINES[mt], color=COLORS[mt],
                     label=mt, linewidth=1.8)
        glob_plotted = True
    if glob_plotted:
        ax_glob.set_title(f'Global attribution drift ({REPORT_EXPLAINER})')
        ax_glob.set_ylabel('1 − RBO')
        ax_glob.legend(fontsize=8)
    else:
        ax_glob.set_title(f'Global attribution drift — not defined for {REPORT_EXPLAINER}')
        ax_glob.text(0.5, 0.5, 'n/a', ha='center', va='center',
                     transform=ax_glob.transAxes, fontsize=14, color='gray')
    ax_glob.set_xticks(x_idx)
    ax_glob.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=7)

    stoch_plotted = False
    for mt in PRESENT:
        dft = df_all[df_all['model_type'] == mt].sort_values('pair_id')
        stoch_col = 'explainer_stoch_max_abs'
        if stoch_col not in dft.columns or dft[stoch_col].isna().all():
            continue

        # Log scale; floor zeros to avoid log(0)
        vals = np.maximum(dft[stoch_col].values, 1e-12)
        ax_stoch.semilogy(_x_local(dft), vals,
                          MARKERS[mt] + LINES[mt], color=COLORS[mt],
                          label=mt, linewidth=1.6)
        stoch_plotted = True
    ax_stoch.set_title('Explainer stochasticity (max |Δφ|)')
    ax_stoch.set_ylabel('Max abs diff (log)')
    if stoch_plotted:
        ax_stoch.legend(fontsize=7)
    else:
        ax_stoch.text(0.5, 0.5, 'no data', ha='center', va='center',
                      transform=ax_stoch.transAxes, fontsize=12, color='gray')
    ax_stoch.set_xticks(x_idx)
    ax_stoch.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=7)

    plt.suptitle(
        f'Temporal explanation-drift diagnostic profile — {REPORT_EXPLAINER}',
        fontsize=14, y=0.998,
    )
    out_path = COMBINED_DIR / f'drift_dashboard_{REPORT_EXPLAINER}.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    print(f'Saved {out_path.name}')
    plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Full-window LR coefficient comparison (transparent reference)
#
# For each pair, plots the standardised coefficient vector of the deterministic
# full-window LR fit on window A vs window B. This replaces a cross-window
# drift score for LR: rather than collapsing the comparison into a single
# distance, both vectors are shown directly.
#
# Note: each side has its own scaler (scaler_A.joblib / scaler_B.joblib).
# The coefficients are therefore in different standardised bases. They are
# directly comparable for sign and qualitative magnitude only; absolute
# magnitudes between A and B should not be over-interpreted.
# ─────────────────────────────────────────────────────────────────────────

LR_DIR_BASE = MODEL_DIR_BASE / 'logreg'

# Pick the top-K most-different features across pairs to keep the plot legible.
TOP_K_FEATURES = 15

available_pairs = []
coef_A_stack = []
coef_B_stack = []
for p in pairs:
    pid = p['pair_id']
    pair_dir = LR_DIR_BASE / f'pair_{pid:02d}'
    a_path = pair_dir / 'coef_A_full.npy'
    b_path = pair_dir / 'coef_B_full.npy'
    if not (a_path.exists() and b_path.exists()):
        continue
    coef_A_stack.append(np.load(a_path))
    coef_B_stack.append(np.load(b_path))
    available_pairs.append(p)

if not available_pairs:
    print('No full-window LR coefficients found — skipping LR reference plot.')
else:
    coef_A_stack = np.array(coef_A_stack)   # (n_pairs, p)
    coef_B_stack = np.array(coef_B_stack)   # (n_pairs, p)

    # Rank features by max-over-pairs absolute coefficient gap.
    gap = np.abs(coef_A_stack - coef_B_stack)
    feat_score = gap.max(axis=0)
    top_idx = np.argsort(-feat_score)[:TOP_K_FEATURES]

    n_pairs = len(available_pairs)
    fig, axes = plt.subplots(
        nrows=n_pairs, ncols=1,
        figsize=(11, 2.0 + 1.4 * n_pairs),
        sharex=True,
    )
    if n_pairs == 1:
        axes = [axes]

    y = np.arange(len(top_idx))
    for ax, p, cA, cB in zip(axes, available_pairs, coef_A_stack, coef_B_stack):
        ax.barh(y - 0.20, cA[top_idx], height=0.4,
                color='steelblue', label=f"A end {p['step_label_A']}")
        ax.barh(y + 0.20, cB[top_idx], height=0.4,
                color='darkorange', label=f"B end {p['step_label_B']}")
        ax.axvline(0.0, color='grey', linewidth=0.6)
        ax.set_yticks(y)
        ax.set_yticklabels([feature_names[i] for i in top_idx], fontsize=7)
        ax.invert_yaxis()
        ax.set_title(f"Pair {p['pair_id']:02d}: full-window LR coefficients (standardised)",
                     fontsize=10)
        ax.legend(fontsize=7, loc='lower right')

    axes[-1].set_xlabel('Standardised coefficient')
    plt.tight_layout()
    out_path = COMBINED_DIR / 'lr_full_window_coef_comparison.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    print(f'Saved {out_path.name}')
    plt.show()

## Per-feature SHAP drift profile

Shows which features shift most in global importance across the timeline.

In [ ]:
# Per-feature attribution over time. Set REPORT_MT to select which
# model_type to visualise. Default: XGBoost + SHAP.
REPORT_MT  = 'xgboost'
if REPORT_MT not in MODEL_TYPES:
    raise ValueError(f'Unknown REPORT_MT: {REPORT_MT!r}')

# Per-feature global SHAP profile is reported for SHAP only — LIME's local-surrogate
# weights are not aggregated into a global ranking in this thesis.
attr_dir_mt = SHAP_DIR_BASE / REPORT_MT
file_A      = 'shap_A.npy'

global_imp_matrix = []
pair_labels = []
for p in pairs:
    pid = p['pair_id']
    ap  = attr_dir_mt / f'pair_{pid:02d}' / file_A
    if not ap.exists():
        continue
    attr_A = np.load(ap)
    phi_bar_A = attr_A.mean(axis=0)
    g_A = np.abs(phi_bar_A).mean(axis=0)
    global_imp_matrix.append(g_A)
    pair_labels.append(p['step_label_A'])

if global_imp_matrix:
    imp_arr = np.array(global_imp_matrix)
    feat_variance = imp_arr.std(axis=0)
    top_feat_idx = np.argsort(-feat_variance)[:15]

    fig, ax = plt.subplots(figsize=(12, 5))
    for j in top_feat_idx:
        ax.plot(range(len(pair_labels)), imp_arr[:, j], marker='o',
                label=feature_names[j], linewidth=1.5)
    ax.set_title(f'Global SHAP importance over time — top 15 variable features  ({REPORT_MT})')
    ax.set_xlabel('Window pair (A end date)')
    ax.set_ylabel('Mean |SHAP|')
    ax.set_xticks(range(len(pair_labels)))
    ax.set_xticklabels(pair_labels, rotation=30, ha='right', fontsize=7)
    ax.legend(loc='upper right', fontsize=7, ncol=2)
    plt.tight_layout()
    out_dir = RESULTS_DIR_BASE / REPORT_MT
    out_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_dir / 'feature_importance_over_time_shap.png', dpi=150)
    plt.show()
else:
    print(f'No SHAP attribution files found for {REPORT_MT}.')

## Summary statistics

In [ ]:
numeric_cols = [
    'pr_auc_A', 'pr_auc_B', 'delta_perf',
    'delta_X', 'delta_Y',
    'loc_cos', 'base_cos_A', 'base_cos_B',
    'loc_rbo', 'base_rbo_A', 'base_rbo_B',
    'global_rbo', 'explainer_stoch_max_abs', 'explainer_stoch_median_abs',
]

if not combined_df.empty:
    for (mt, exp), group_df in combined_df.groupby(['model_type', 'explainer']):
        avail_cols = [c for c in numeric_cols if c in group_df.columns]
        summary = group_df[avail_cols].describe().T[['mean', 'std', 'min', 'max']]
        print(f'\n=== {mt} / {exp} — overall summary ({len(group_df)} pairs) ===')
        print(summary.round(4).to_string())

        # LR rows show NaN for cross-window drift columns by design — they are a
        # transparent reference, not a post-hoc explanation drift stream.

        # Side-by-side mean local drift vs mean baselines (no ratio).
        # Helps eyeball whether dynamic drift sits above the within-window noise floor.
        if all(c in group_df.columns for c in
               ['loc_cos', 'base_cos_A', 'base_cos_B', 'loc_rbo', 'base_rbo_A', 'base_rbo_B']):
            print(
                f"  mean loc_cos = {group_df['loc_cos'].mean():.4f}  |  "
                f"mean baseline_A = {group_df['base_cos_A'].mean():.4f}  |  "
                f"mean baseline_B = {group_df['base_cos_B'].mean():.4f}"
            )
            print(
                f"  mean loc_rbo = {group_df['loc_rbo'].mean():.4f}  |  "
                f"mean baseline_A = {group_df['base_rbo_A'].mean():.4f}  |  "
                f"mean baseline_B = {group_df['base_rbo_B'].mean():.4f}"
            )
